# Mental Rotation — Fine-Tuned Spatial VLMs

Follow-up to `mental-rotation-fixes.ipynb` and `mental-rotation-advanced.ipynb`.
Tests **spatial-reasoning fine-tuned models** against our mental rotation benchmark.

### Background

Mental rotation remains one of the hardest tasks for current VLMs. The
[SpinBench](https://spinbench25.github.io/) benchmark (ICLR 2026) found that
even top models exhibit "poor rotational understanding" and "strong egocentric
bias", with humans scoring 91.2% while models hover near chance on difficult
rotation tasks. [TransformEval](https://openreview.net/pdf?id=up2LD7vVdW)
reported that models achieve < 40% on basic 2D rotations and ~30% on 3D mental
rotation tasks directly paralleling Shepard & Metzler (1971).

Our prior notebooks showed that prompt engineering alone reaches ~63% with
Qwen2.5-VL-3B (self-consistency k=5 on the elimination prompt). The hypothesis
here is that models **fine-tuned for spatial reasoning** can do better out of the
box, even without task-specific training on mental rotation stimuli.

### Models tested

| Model | Params | Base | Key technique | Reference |
|---|---|---|---|---|
| Qwen2.5-VL-3B | 3B | — | Baseline (no spatial fine-tuning) | [Qwen](https://qwenlm.github.io/blog/qwen2.5-vl/) |
| SpaceThinker-3B | 3B | Qwen2.5-VL-3B | LoRA on synthetic spatial reasoning traces (VQASynth) | [HF](https://huggingface.co/remyxai/SpaceThinker-Qwen2.5VL-3B) |
| SpaceOm-3B | 3B | Qwen2.5-VL-3B | SpaceThinker + longer reasoning + robotics data | [HF](https://huggingface.co/remyxai/SpaceOm) |
| SpatialThinker-3B (Oxford) | 3B | Qwen2.5-VL-3B | RL with dense spatial rewards + scene graph grounding (7K samples) | [GitHub](https://github.com/hunarbatra/SpatialThinker) |
| Spatial-SSRL-3B | 3B | Qwen2.5-VL-3B | Self-supervised RL with 5 pretext tasks (flip detection, depth ordering) | [GitHub](https://github.com/InternLM/Spatial-SSRL) |

### Prompt strategy

Each model is tested with **two** prompt approaches (greedy, no self-consistency):

1. **Baseline (S2 elimination)**: Short system prompt + elimination framing
   ("identify the mirror, pick the other"). `max_new_tokens=128`.
2. **Paper-recommended**: The prompt style described in each model's
   documentation, with structured reasoning tags and longer generation.

| Model | Paper prompt style | Reasoning tags | max_tokens |
|---|---|---|---|
| Qwen2.5-VL-3B | *(baseline only — no paper prompt)* | — | 128 |
| SpaceThinker / SpaceOm | VL-Thinking system prompt | `<think>` → `<answer>` | 1024 |
| SpatialThinker (Oxford) | Observe → scene graph → think → answer | `<observe>` → `<scene>` → `<think>` → `<answer>` | 1024 |
| Spatial-SSRL | Think → boxed answer | `<think>` → `\boxed{}` | 2048 |

### Prerequisites

```bash
pip install transformers accelerate pillow scipy tqdm
```

In [ ]:
from __future__ import annotations

import base64, csv, json, mimetypes, os, random, re
from collections import Counter
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from PIL import Image

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data").is_dir() and (p / "src").is_dir()),
    Path.cwd().parent.parent,
)

LABELS = ["A", "B"]
CHANCE = 0.5
DATA_ROOT = ROOT / "data"
MANIFEST_PATH = DATA_ROOT / "assets" / "manifest.csv"
RESULTS_DIR = ROOT / "results" / "prompt_optimization" / "mental-rotation" / "finetuned"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

## Data loading

In [ ]:
def _detect_image_dir() -> Path:
    assets = DATA_ROOT / "assets"
    for child in sorted(assets.iterdir(), reverse=True):
        candidate = child / "visual" / "mental-rotation"
        if candidate.is_dir() and any(candidate.iterdir()):
            return candidate
    raise FileNotFoundError("No mental-rotation images found.")


def _build_image_index(directory: Path) -> dict[str, Path]:
    index: dict[str, Path] = {}
    if not directory.is_dir():
        return index
    for path in directory.iterdir():
        if path.is_file():
            index[path.stem] = path
            index[path.name] = path
    return index


def _extract_angle(item_uid: str) -> Optional[int]:
    m = re.search(r"_(\d{3})_", item_uid)
    if m:
        return int(m.group(1))
    if item_uid.endswith("-000") or "_000" in item_uid or item_uid.startswith(("Rn-000", "Rp-000")):
        return 0
    return None


def load_trials() -> list[dict]:
    rows = []
    with open(MANIFEST_PATH, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["task"] == "mental-rotation":
                rows.append(row)

    trials = []
    for row in rows:
        answer = str(row["answer"]).strip()
        alternatives = [a.strip() for a in row["response_alternatives"].split(",") if a.strip()]
        all_options = [answer] + alternatives
        rng = random.Random(row["item_uid"])
        rng.shuffle(all_options)
        correct_idx = all_options.index(answer)
        correct_label = LABELS[correct_idx] if correct_idx < len(LABELS) else "?"

        option_image_paths = []
        missing = False
        for option in all_options:
            path = IMAGE_INDEX.get(option.strip())
            if path is None:
                missing = True
                break
            option_image_paths.append(str(path))

        prompt_image_stem = str(row.get("prompt_image", "")).strip()
        context_image_paths = []
        if prompt_image_stem and prompt_image_stem not in {"NA", "nan", "TODO", ""}:
            path = IMAGE_INDEX.get(prompt_image_stem)
            if path is None:
                missing = True
            else:
                context_image_paths.append(str(path))

        if missing:
            continue

        trials.append({
            "item_uid": row["item_uid"],
            "trial_type": str(row.get("trial_type", "")).strip(),
            "angle": _extract_angle(row["item_uid"]),
            "full_prompt": row.get("full_prompt", ""),
            "options": all_options,
            "option_image_paths": option_image_paths,
            "context_image_paths": context_image_paths,
            "correct_label": correct_label,
        })
    return trials


IMAGE_DIR = _detect_image_dir()
IMAGE_INDEX = _build_image_index(IMAGE_DIR)
TRIALS = load_trials()
print(f"Image dir: {IMAGE_DIR} ({len(IMAGE_INDEX) // 2} images)")
print(f"Loaded {len(TRIALS)} trials")

## Model registry

Each entry maps a short name to a HuggingFace model ID. Toggle which models to
run by commenting/uncommenting. All Qwen2.5-VL-based models share the same
inference backend; InternVL-based models (SpatialReasoner-R1) need a separate
code path.

In [ ]:
MODEL_REGISTRY: dict[str, dict] = {
    "Qwen2.5-VL-3B": {
        "hf_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "backend": "qwen",
        "params": "3B",
        "note": "Baseline (no spatial fine-tuning)",
    },
    "SpaceThinker-3B": {
        "hf_id": "remyxai/SpaceThinker-Qwen2.5VL-3B",
        "backend": "qwen",
        "params": "3B",
        "note": "LoRA fine-tuned on VQASynth spatial data (12K samples)",
    },
    "SpaceOm-3B": {
        "hf_id": "remyxai/SpaceOm",
        "backend": "qwen",
        "params": "3B",
        "note": "SpaceThinker successor: longer traces + robotics data",
    },
    "SpatialThinker-3B": {
        "hf_id": "OX-PIXL/SpatialThinker-3B",
        "backend": "qwen",
        "params": "3B",
        "note": "RL with dense spatial rewards + scene graph grounding (Oxford, NeurIPS 2025 WS)",
    },
    "Spatial-SSRL-3B": {
        "hf_id": "internlm/Spatial-SSRL-3B",
        "backend": "qwen",
        "params": "3B",
        "note": "Self-supervised RL: flip detection, depth ordering, patch tasks (CVPR 2026)",
    },
    # Requires downloading from HF: hongxingli/SpatialLadder-3B
    # Uses Qwen2.5-VL-3B base + 3-stage progressive training + RL
    # "SpatialLadder-3B": {
    #     "hf_id": "hongxingli/SpatialLadder-3B",
    #     "backend": "qwen",
    #     "params": "3B",
    #     "note": "Progressive spatial training + GRPO (ICLR 2026)",
    # },
    # 7B models — need more VRAM and/or custom training
    # SpatialDreamer: no pretrained checkpoint on HF; must be trained via verl
    # from their code at github.com/mengcaopku/SpatialDreamer
    # Achieves SAT 93.9%, MindCube 84.9% with GeoPO on Qwen2.5-VL-7B
    # InternVL-based — requires different loading code
    # "SpatialReasoner-R1-4B": {
    #     "hf_id": "PLAN-Lab/SpatialReasoner-R1",
    #     "backend": "internvl",
    #     "params": "4B",
    #     "note": "Fine-grained DPO on spatial reasoning (NeurIPS 2025)",
    # },
}

ACTIVE_MODELS = list(MODEL_REGISTRY.keys())
print(f"Active models ({len(ACTIVE_MODELS)}):")
for name in ACTIVE_MODELS:
    info = MODEL_REGISTRY[name]
    print(f"  {name:24s}  {info['params']:>4s}  {info['note']}")

## Model loading and generation

In [ ]:
_model_cache: dict[str, tuple] = {}


def load_hf_model(model_id: str):
    if model_id in _model_cache:
        return _model_cache[model_id]
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText

    device = "cuda" if torch.cuda.is_available() else (
        "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
    )
    dtype = torch.bfloat16
    processor = AutoProcessor.from_pretrained(model_id, padding_side="left")
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=dtype, attn_implementation="sdpa",
    ).to(device)
    model.eval()
    result = (model, processor, dtype, device)
    _model_cache[model_id] = result
    return result


def generate_hf(
    model, processor, dtype, device,
    prompt_text: str, image_paths: list[str],
    max_new_tokens: int = 128,
    system_prompt: str | None = None,
    do_sample: bool = False,
    temperature: float = 1.0,
) -> str:
    import torch

    pil_images = [Image.open(p).convert("RGB") for p in image_paths]

    content: list[dict] = []
    if pil_images and re.search(r"<image\d+>", prompt_text):
        parts = re.split(r"(<image\d+>)", prompt_text)
        for part in parts:
            m = re.match(r"<image(\d+)>", part)
            if m:
                idx = int(m.group(1))
                if idx < len(pil_images):
                    content.append({"type": "image", "image": pil_images[idx]})
            elif part.strip():
                content.append({"type": "text", "text": part.strip()})
    else:
        for img in pil_images:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": prompt_text})

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": content})

    try:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    inputs = processor(
        text=[text],
        images=pil_images if pil_images else None,
        return_tensors="pt",
        padding=True,
    ).to(device)

    gen_kwargs = dict(max_new_tokens=max_new_tokens)
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen_kwargs["do_sample"] = False

    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    generated_ids = output_ids[:, input_len:]
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

## Answer parsing and statistics

In [ ]:
def parse_answer(text: str, option_labels: list[str] = LABELS) -> Optional[str]:
    """Parse a model response into one of the option labels (A/B).

    Handles multiple output formats from different models:
    - Plain letter: "A" or "B"
    - JSON: {"answer": "A"}
    - <answer> tags: <answer> (A) rotated </answer>
    - \\boxed{}: \\boxed{A}
    - Natural language: "the answer is A"
    """
    raw = text.strip()
    labels_upper = [la.upper() for la in option_labels]

    # Strip <think>...</think> blocks (used by SpaceThinker, SpaceOm, Spatial-SSRL, OX-PIXL)
    cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    # Strip <observe>...</observe> and <scene>...</scene> blocks (OX-PIXL)
    cleaned = re.sub(r"<observe>.*?</observe>", "", cleaned, flags=re.DOTALL).strip()
    cleaned = re.sub(r"<scene>.*?</scene>", "", cleaned, flags=re.DOTALL).strip()

    # 1. <answer> tags (OX-PIXL SpatialThinker, remyxai SpaceThinker/SpaceOm)
    m = re.search(r"<answer>\s*\(?([A-B])\)?[^<]*</answer>", cleaned, re.IGNORECASE)
    if m and m.group(1).upper() in labels_upper:
        return m.group(1).upper()

    # 2. \boxed{} (Spatial-SSRL)
    m = re.search(r"\\boxed\{([A-B])\}", cleaned, re.IGNORECASE)
    if m and m.group(1).upper() in labels_upper:
        return m.group(1).upper()

    # 3. JSON {"answer": "A"}
    try:
        parsed = json.loads(cleaned)
        ans = parsed.get("answer", "").strip().upper()
        if ans in labels_upper:
            return ans
    except (json.JSONDecodeError, AttributeError):
        pass

    m = re.search(r'"answer"\s*:\s*"?([A-Z])\b', cleaned, re.IGNORECASE)
    if m and m.group(1).upper() in labels_upper:
        return m.group(1).upper()

    # 4. Natural language patterns
    for pat in (
        r"(?:the\s+)?(?:correct\s+)?answer\s+is\s+\(?([A-B])\)?\b",
        r"option\s+\(?([A-B])\)?\b",
        r"(?:choose|select|pick)\s+\(?([A-B])\)?\b",
        r"\(([A-B])\)",
    ):
        m = re.search(pat, cleaned, re.IGNORECASE)
        if m and m.group(1).upper() in labels_upper:
            return m.group(1).upper()

    # 5. Bare letter
    if cleaned.upper() in labels_upper:
        return cleaned.upper()

    for label in option_labels:
        if cleaned.upper().startswith(label.upper()):
            rest = cleaned[len(label):]
            if not rest or rest[0] in " .),:;\n":
                return label.upper()

    # 6. Last-sentence fallback
    for sentence in reversed(re.split(r"[.!?\n]", cleaned)):
        for label in option_labels:
            if re.search(rf"\b{re.escape(label)}\b", sentence, re.IGNORECASE):
                return label.upper()

    return None


def _binom_p(correct: int, n: int) -> float:
    if n == 0:
        return 1.0
    return stats.binomtest(correct, n, CHANCE, alternative="greater").pvalue

## Prompt strategies

For each model we test **two** prompt approaches:

1. **Baseline (S2 elimination)**: The best prompt from `mental-rotation-fixes.ipynb` — identical
   across all models, with our standard system prompt. This isolates the effect of model fine-tuning.

2. **Paper-recommended**: The prompting style documented in each model's paper or HF card.
   These models were trained with specific prompt templates, so using them should unlock the
   model's intended reasoning behavior.

| Model | Paper prompt style | Key difference |
|---|---|---|
| **Qwen2.5-VL-3B** | Same as baseline | No special prompt style |
| **SpaceThinker-3B** (remyxai) | VL-Thinking system prompt: `<think>...</think>` + `<answer>...</answer>` | Enables long reasoning chain (max 1024 tokens) |
| **SpaceOm-3B** (remyxai) | Same as SpaceThinker (longer traces) | Same VL-Thinking prompt, longer outputs |
| **SpatialThinker-3B** (Oxford) | `<observe>` → `<scene>` (scene graph) → `<think>` → `<answer>` | Structured multi-step spatial template with image size |
| **Spatial-SSRL-3B** (InternLM) | `<think>...</think>` + `\boxed{}` | RLVR-style reasoning with boxed answer |

In [ ]:
# ── Baseline prompt (shared across all models) ───────────────────────────────

BASELINE_SYSTEM = (
    "You are a visual reasoning assistant. "
    "Answer with only a single letter: A or B. Do not explain."
)

ELIMINATION_TEXT = (
    "The first image is the reference shape. "
    "The second image is option A. The third image is option B.\n\n"
    "One option is the SAME shape rotated to a different angle. "
    "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
    "First, identify which option looks like a mirror/flip of the reference.\n"
    "Then pick the OTHER option — the one that is simply rotated.\n\n"
    "Answer with one letter: A or B."
)


def prompt_baseline(trial: dict) -> dict:
    """Baseline S2 elimination prompt — identical for all models."""
    return {
        "prompt_text": ELIMINATION_TEXT,
        "image_paths": trial["context_image_paths"] + trial["option_image_paths"],
        "system_prompt": BASELINE_SYSTEM,
        "max_new_tokens": 128,
    }


# ── Paper-recommended prompts (model-specific) ───────────────────────────────

# SpaceThinker / SpaceOm (remyxai): VL-Thinking with <think> + <answer> tags
REMYX_SYSTEM = (
    "You are VL-Thinking, a helpful assistant with excellent reasoning ability. "
    "You should first think about the reasoning process and then provide the answer. "
    "Use <think>...</think> and <answer>...</answer> tags."
)

REMYX_TASK_TEXT = (
    "The first image is the reference shape. "
    "The second image is option A. The third image is option B.\n\n"
    "One option is the SAME shape rotated to a different angle. "
    "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
    "Think step by step about the spatial structure of each shape. "
    "Identify asymmetric features and how they change under rotation vs mirroring. "
    "Then determine which option is the rotated copy (not the mirror).\n\n"
    "Answer with A or B inside <answer> tags."
)


def prompt_remyx_paper(trial: dict) -> dict:
    """SpaceThinker/SpaceOm paper prompt: VL-Thinking with structured reasoning."""
    return {
        "prompt_text": REMYX_TASK_TEXT,
        "image_paths": trial["context_image_paths"] + trial["option_image_paths"],
        "system_prompt": REMYX_SYSTEM,
        "max_new_tokens": 1024,
    }


# SpatialThinker (OX-PIXL Oxford): <observe> → <scene> → <think> → <answer>
OXPIXL_TEMPLATE = (
    "You FIRST observe the image in <observe> </observe> tags, "
    "then visualise the relevant scene graph in <scene> </scene> tags, "
    "followed by thinking about the reasoning process as an internal monologue "
    "within <think> </think> tags and then provide the final answer. "
    "The final answer MUST BE put within <answer> </answer> tags, "
    "and only return the final choice including the correct option and answer "
    "within the answer tags, e.g., <answer> (A) rotated </answer>.\n\n"
)


def prompt_oxpixl_paper(trial: dict) -> dict:
    """OX-PIXL SpatialThinker paper prompt: observe → scene graph → think → answer."""
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    # Get image dimensions for the template (uses first image as representative)
    try:
        img = Image.open(image_paths[0])
        w, h = img.size
        img.close()
    except Exception:
        w, h = 512, 512

    task = (
        f"Image size: {w} x {h}\n\n"
        "The first image is the reference shape. "
        "The second image is option (A). The third image is option (B).\n\n"
        "One option is the SAME shape rotated to a different angle. "
        "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
        "Which option is the rotated copy (not the mirror)? Answer (A) or (B)."
    )
    return {
        "prompt_text": OXPIXL_TEMPLATE + task,
        "image_paths": image_paths,
        "system_prompt": None,
        "max_new_tokens": 1024,
    }


# Spatial-SSRL (InternLM): <think> reasoning + \boxed{} answer
SSRL_FORMAT_SUFFIX = (
    "\n You FIRST think about the reasoning process as an internal monologue "
    "and then provide the final answer. The reasoning process MUST BE enclosed "
    "within <think> </think> tags. The final answer MUST BE put in \\boxed{}."
)


def prompt_ssrl_paper(trial: dict) -> dict:
    """Spatial-SSRL paper prompt: think + boxed answer (RLVR-style)."""
    task = (
        "The first image is the reference shape. "
        "The second image is option A. The third image is option B.\n\n"
        "One option is the SAME shape rotated to a different angle. "
        "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
        "Which option is the rotated copy (not the mirror)? Answer A or B."
        + SSRL_FORMAT_SUFFIX
    )
    return {
        "prompt_text": task,
        "image_paths": trial["context_image_paths"] + trial["option_image_paths"],
        "system_prompt": None,
        "max_new_tokens": 2048,
    }


# ── Registry: which paper prompt to use for each model ───────────────────────

PAPER_PROMPT_FN: dict[str, callable] = {
    "Qwen2.5-VL-3B": prompt_baseline,
    "SpaceThinker-3B": prompt_remyx_paper,
    "SpaceOm-3B": prompt_remyx_paper,
    "SpatialThinker-3B": prompt_oxpixl_paper,
    "Spatial-SSRL-3B": prompt_ssrl_paper,
}

## Evaluation runners

In [ ]:
# Experiment summary (printed, not executed)
print("Experiment design:")
print(f"  Models: {len(ACTIVE_MODELS)}")
for m in ACTIVE_MODELS:
    paper_fn = PAPER_PROMPT_FN.get(m, prompt_baseline)
    has_paper = paper_fn is not prompt_baseline
    strategies = ["baseline"]
    if has_paper:
        strategies.append("paper")
    if RUN_SELF_CONSISTENCY:
        strategies.append("baseline_SC5")
        if has_paper:
            strategies.append("paper_SC5")
    print(f"    {m:24s} → {', '.join(strategies)}")
n_strategies = sum(
    2 + (2 if PAPER_PROMPT_FN.get(m, prompt_baseline) is not prompt_baseline else 0)
    if RUN_SELF_CONSISTENCY else
    1 + (1 if PAPER_PROMPT_FN.get(m, prompt_baseline) is not prompt_baseline else 0)
    for m in ACTIVE_MODELS
)
print(f"  Total strategy × model runs: {n_strategies}")

In [ ]:
from tqdm.notebook import tqdm


def run_strategy(
    strategy_name: str,
    generate_fn,
    trials: list[dict],
    prompt_fn: callable,
) -> list[dict]:
    """Run a single prompt strategy (greedy) over all trials."""
    results = []
    for trial in tqdm(trials, desc=strategy_name):
        cfg = prompt_fn(trial)
        try:
            response = generate_fn(
                cfg["prompt_text"], cfg["image_paths"],
                max_new_tokens=cfg.get("max_new_tokens", 128),
                system_prompt=cfg.get("system_prompt"),
                do_sample=False,
            )
        except Exception as e:
            response = f"ERROR: {e}"

        predicted = parse_answer(response)
        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "raw_response": response,
            "strategy": strategy_name,
        })
    return results


def run_self_consistent(
    strategy_name: str,
    generate_fn,
    trials: list[dict],
    prompt_fn: callable,
    n_samples: int = 5,
    temperature: float = 0.7,
) -> list[dict]:
    """Majority vote over n_samples temperature-sampled responses."""
    results = []
    for trial in tqdm(trials, desc=f"{strategy_name} (SC k={n_samples})"):
        cfg = prompt_fn(trial)
        votes = []
        raw_responses = []
        for _ in range(n_samples):
            try:
                resp = generate_fn(
                    cfg["prompt_text"], cfg["image_paths"],
                    max_new_tokens=cfg.get("max_new_tokens", 128),
                    system_prompt=cfg.get("system_prompt"),
                    do_sample=True,
                    temperature=temperature,
                )
            except Exception as e:
                resp = f"ERROR: {e}"
            raw_responses.append(resp)
            pred = parse_answer(resp)
            if pred:
                votes.append(pred)

        if votes:
            counter = Counter(votes)
            predicted = counter.most_common(1)[0][0]
        else:
            predicted = None

        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "raw_response": " | ".join(raw_responses),
            "n_votes": len(votes),
            "vote_distribution": dict(Counter(votes)),
            "strategy": f"{strategy_name}_SC5",
        })
    return results

## Configuration

In [ ]:
LIMIT = None  # Set to e.g. 10 for a smoke test
RUN_SELF_CONSISTENCY = True  # Set to False to skip SC runs (5x slower)

trials_to_run = TRIALS[:LIMIT] if LIMIT else TRIALS
print(f"Trials: {len(trials_to_run)}")
print(f"Self-consistency: {'enabled (k=5)' if RUN_SELF_CONSISTENCY else 'disabled'}")

## Run all models

Models are loaded one at a time to fit in memory. After evaluation, the model is
removed from the cache to free VRAM/RAM before loading the next one.

In [ ]:
import gc

all_results: dict[str, list[dict]] = {}

for model_name in ACTIVE_MODELS:
    info = MODEL_REGISTRY[model_name]
    hf_id = info["hf_id"]

    if info["backend"] != "qwen":
        print(f"\nSkipping {model_name} — {info['backend']} backend not implemented yet.")
        continue

    print(f"\n{'='*60}")
    print(f"Loading {model_name} ({hf_id})...")
    print(f"{'='*60}")

    model, processor, dtype, device = load_hf_model(hf_id)

    def _gen(prompt, image_paths, max_new_tokens=128, system_prompt=None,
             do_sample=False, temperature=1.0,
             _m=model, _p=processor, _dt=dtype, _dv=device):
        return generate_hf(
            _m, _p, _dt, _dv,
            prompt, image_paths, max_new_tokens,
            system_prompt=system_prompt,
            do_sample=do_sample,
            temperature=temperature,
        )

    # ── Strategy 1: Baseline elimination prompt (greedy) ──────────────────
    key_base = f"{model_name}_baseline"
    all_results[key_base] = run_strategy(key_base, _gen, trials_to_run, prompt_baseline)

    # ── Strategy 2: Paper-recommended prompt (greedy) ─────────────────────
    paper_fn = PAPER_PROMPT_FN.get(model_name, prompt_baseline)
    if paper_fn is not prompt_baseline:
        key_paper = f"{model_name}_paper"
        all_results[key_paper] = run_strategy(key_paper, _gen, trials_to_run, paper_fn)

    # ── Strategy 3: Baseline + self-consistency k=5 ───────────────────────
    if RUN_SELF_CONSISTENCY:
        key_sc = f"{model_name}_baseline"
        all_results[f"{key_sc}_SC5"] = run_self_consistent(
            key_sc, _gen, trials_to_run, prompt_baseline,
            n_samples=5, temperature=0.7,
        )

    # ── Strategy 4: Paper prompt + self-consistency k=5 ───────────────────
    if RUN_SELF_CONSISTENCY and paper_fn is not prompt_baseline:
        key_psc = f"{model_name}_paper"
        all_results[f"{key_psc}_SC5"] = run_self_consistent(
            key_psc, _gen, trials_to_run, paper_fn,
            n_samples=5, temperature=0.7,
        )

    # Free memory before loading next model
    del model, processor
    _model_cache.pop(hf_id, None)
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

print(f"\nAll runs complete. {len(all_results)} result sets.")

## Summary results

In [ ]:
summary_rows = []
for name in sorted(all_results):
    results = all_results[name]
    n = len(results)
    correct = sum(1 for r in results if r["is_correct"])
    parsed = sum(1 for r in results if r["predicted_label"] is not None)
    acc = correct / n if n else 0
    p_val = _binom_p(correct, n)
    summary_rows.append({
        "Model + Strategy": name,
        "N": n,
        "Correct": correct,
        "Accuracy": acc,
        "Parse %": parsed / n if n else 0,
        "p-value": p_val,
        "Sig (p<.05)": "Yes" if p_val < 0.05 else "No",
    })

df_summary = pd.DataFrame(summary_rows).sort_values("Accuracy", ascending=False)
display(
    df_summary.style
    .format({"Accuracy": "{:.1%}", "Parse %": "{:.1%}", "p-value": "{:.4f}"})
    .applymap(
        lambda v: "background-color: #c6efce" if v == "Yes" else "",
        subset=["Sig (p<.05)"],
    )
    .hide(axis="index")
)

## Breakdown: 2D silhouettes vs 3D blocks

In [ ]:
TYPE_LABELS = {"2": "2D silhouettes", "3": "3D blocks"}

breakdown_rows = []
for name in sorted(all_results):
    for tt in ("2", "3"):
        subset = [r for r in all_results[name] if r["trial_type"] == tt]
        if not subset:
            continue
        n = len(subset)
        correct = sum(1 for r in subset if r["is_correct"])
        acc = correct / n
        p_val = _binom_p(correct, n)
        breakdown_rows.append({
            "Model + Strategy": name,
            "Type": TYPE_LABELS.get(tt, tt),
            "N": n,
            "Accuracy": acc,
            "p-value": p_val,
        })

df_bd = pd.DataFrame(breakdown_rows)
pivot = df_bd.pivot_table(
    index="Model + Strategy", columns="Type", values="Accuracy", aggfunc="first",
)
display(
    pivot.style
    .format("{:.1%}")
    .background_gradient(cmap="RdYlGn", vmin=0.35, vmax=0.80)
)

## Response bias check

In [ ]:
bias_rows = []
for name in sorted(all_results):
    results = all_results[name]
    parsed = [r for r in results if r["predicted_label"] is not None]
    if not parsed:
        continue
    counts = {"A": 0, "B": 0}
    for r in parsed:
        label = r["predicted_label"]
        if label in counts:
            counts[label] += 1
    n = len(parsed)
    expected = n / 2
    chi2 = sum((v - expected) ** 2 / expected for v in counts.values())
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
    bias_rows.append({
        "Model + Strategy": name,
        "N parsed": n,
        "A count": counts["A"],
        "B count": counts["B"],
        "A %": f"{counts['A'] / n:.0%}",
        "chi2 p": f"{p_val:.3f}",
        "Biased (p<.05)": "Yes" if p_val < 0.05 else "No",
    })

display(pd.DataFrame(bias_rows).style.hide(axis="index"))

## Accuracy by rotation angle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (tt, tt_label) in zip(axes, [("2", "2D silhouettes"), ("3", "3D blocks")]):
    for name in sorted(all_results):
        results = all_results[name]
        subset = [r for r in results if r["trial_type"] == tt and r["angle"] is not None]
        if not subset:
            continue
        df_tmp = pd.DataFrame(subset)
        angle_acc = df_tmp.groupby("angle")["is_correct"].mean()
        ax.plot(angle_acc.index, angle_acc.values, "o-", label=name, alpha=0.7, markersize=4)
    ax.axhline(CHANCE, color="gray", linestyle=":", linewidth=1, label="chance")
    ax.set_xlabel("Rotation angle (degrees)")
    ax.set_ylabel("Accuracy")
    ax.set_title(tt_label)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=7, loc="lower left")

fig.suptitle("Fine-tuned models: accuracy by rotation angle", fontsize=13)
plt.tight_layout()
plt.show()

## Model comparison chart

In [ ]:
plot_data = df_summary.copy().sort_values("Accuracy", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(plot_data) * 0.6)))
colors = ["#2ca02c" if sig == "Yes" else "#1f77b4" for sig in plot_data["Sig (p<.05)"]]
bars = ax.barh(plot_data["Model + Strategy"], plot_data["Accuracy"], color=colors)
ax.axvline(CHANCE, color="red", linestyle="--", linewidth=1, label="Chance (50%)")
ax.set_xlabel("Accuracy")
ax.set_title("Mental Rotation: Fine-Tuned Spatial VLMs")
ax.set_xlim(0, 1)

for bar, acc in zip(bars, plot_data["Accuracy"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{acc:.1%}", va="center", fontsize=9)

ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Save results

In [ ]:
for name, results in all_results.items():
    path = RESULTS_DIR / f"{name}.csv"
    df = pd.DataFrame(results)
    for col in ["vote_distribution"]:
        if col in df.columns:
            df[col] = df[col].astype(str)
    df.to_csv(path, index=False)
    print(f"Saved {path.name} ({len(results)} rows)")

summary_path = RESULTS_DIR / "summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"\nSaved {summary_path.name}")

## Experiment results (2026-04-09)

Run on Apple M5 Pro (48 GB), MPS backend. 83 trials, greedy decoding, no
self-consistency. Spatial-SSRL-3B was interrupted during download and excluded.

### Summary

| Model + Strategy | N | Correct | Accuracy | Parse % | p-value | Sig |
|---|---|---|---|---|---|---|
| **SpaceThinker-3B_baseline** | 83 | 49 | **59.0%** | 100% | 0.0619 | |
| **SpaceOm-3B_baseline** | 83 | 49 | **59.0%** | 100% | 0.0619 | |
| **SpatialThinker-3B_baseline** | 83 | 49 | **59.0%** | 98% | 0.0619 | |
| Qwen2.5-VL-3B_baseline | 83 | 45 | 54.2% | 93% | 0.2552 | |
| SpaceOm-3B_paper | 83 | 40 | 48.2% | 99% | 0.6696 | |
| SpatialThinker-3B_paper | 83 | 34 | 41.0% | 100% | 0.9608 | |
| SpaceThinker-3B_paper | 83 | 32 | 38.6% | 100% | 0.9862 | |

**No model reached significance** (one-tailed binomial test vs. 50% chance).

### The 59.0% artifact: position bias, not spatial reasoning

Three of four fine-tuned models scored exactly **49/83 = 59.0%** on the
baseline prompt. This is not a coincidence — it is the dataset base rate of
trials where "B" is the correct answer.

The `mental-rotation-fixes.ipynb` notebook found the same pattern with
Qwen2.5-VL-3B:

| Strategy (from fixes notebook) | Accuracy | A count | B count |
|---|---|---|---|
| S2_elimination | 59.0% | **0** | **83** |
| S5_mirror_only | 59.0% | **0** | **83** |
| S7_imgfirst_elim | 59.0% | **0** | **83** |

All three had **100% B-bias** — the model always answered "B" regardless of
content. The 59.0% accuracy is entirely explained by how many trials happen to
have B as the correct answer.

The baseline prompt (short system prompt + elimination framing) triggers
**position bias collapse** in 3B Qwen-based models: the model anchors to the
last option presented and always picks it. The spatial fine-tuning does not fix
this behavior — SpaceThinker, SpaceOm, and SpatialThinker all exhibit the same
pattern.

### Paper-recommended prompts: longer reasoning hurts

| Model | Baseline | Paper | Delta |
|---|---|---|---|
| SpaceThinker-3B | 59.0%* | 38.6% | −20.4% |
| SpaceOm-3B | 59.0%* | 48.2% | −10.8% |
| SpatialThinker-3B (Oxford) | 59.0%* | 41.0% | −18.0% |

*\* Inflated by B-bias (see above).*

The paper-recommended prompts produce longer reasoning chains (1024 tokens)
but perform **worse** than the biased baseline, and worse than chance. This
suggests that 3B models lack sufficient capacity to reason about mental
rotation through extended text — the model "talks itself out of" initially
plausible answers.

### Key findings

1. **Spatial fine-tuning does not transfer to mental rotation**: Despite being
   trained on spatial reasoning tasks (depth estimation, relative positioning,
   scene understanding), none of these models improve over the base Qwen2.5-VL-3B
   on mental rotation. The fine-tuning targets different spatial sub-skills.

2. **Position bias dominates at 3B scale**: All Qwen2.5-VL-3B-based models
   collapse to always answering "B" when given the short elimination prompt.
   This is a fundamental limitation of the model scale, not the fine-tuning.

3. **Extended reasoning is counterproductive**: Prompts that encourage
   step-by-step spatial reasoning (via `<think>`, `<observe>`, `<scene>` tags)
   actively hurt performance. The models generate plausible-sounding but
   incorrect spatial descriptions.

4. **Mental rotation remains hard for small VLMs**: Even with specialized
   spatial training, 3B models hover at or below chance (50%) on this task.
   The SpinBench and TransformEval findings about "poor rotational
   understanding" are confirmed at this scale.

### Timing

| Run | Duration |
|---|---|
| Qwen2.5-VL-3B_baseline | 7.2 min |
| SpaceThinker-3B_baseline | 5.4 min |
| SpaceThinker-3B_paper | 37.1 min |
| SpaceOm-3B_baseline | 5.8 min |
| SpaceOm-3B_paper | 31.3 min |
| SpatialThinker-3B_baseline | 9.3 min |
| SpatialThinker-3B_paper | 11.2 min |
| **Total (7 runs)** | **~107 min** |

Paper-recommended prompts take 3–7× longer than baseline due to generating
1024 tokens of reasoning (vs. 128 for baseline).

## Notes

### Models not yet tested (require additional setup)

| Model | Why not included | How to add |
|---|---|---|
| **Spatial-SSRL-3B** | Run interrupted during experiment (model downloaded but evaluation did not complete) | Already in `MODEL_REGISTRY`; re-run the evaluation loop |
| **SpatialDreamer-7B** | 7B model (needs ~14GB VRAM); RL with active mental imagery + GeoPO; SAT 93.9%, MindCube 84.9% | Uncomment in `MODEL_REGISTRY`; [GitHub](https://github.com/mengcaopku/SpatialDreamer) |
| **STAR-R1-7B** | 7B; **trained on transformation reasoning** (TVR) — 61.4% TAcc vs GPT-4o 23.5%; weights not yet released | Monitor [GitHub](https://github.com/zongzhao23/STAR-R1) for weight release |
| **ViLaSR-7B** | 7B; visual drawing + RL; +18.4% avg across benchmarks (NeurIPS 2025) | [HF](https://huggingface.co/AntResearchNLP/ViLaSR) |
| **SpatialLadder-3B** | Requires `hongxingli/SpatialLadder-3B` download; Qwen2.5-VL-3B base | Uncomment in `MODEL_REGISTRY` |
| **SpatialReasoner-R1** (4B/8B) | InternVL2.5 architecture — needs `lmdeploy` or custom loading | Implement `internvl` backend |
| **Spatial-MLLM-4B** | Dual encoder (2D + VGGT 3D); custom architecture | Follow [THU-SI/Spatial-MLLM](https://github.com/THU-SI/Spatial-MLLM) setup |
| **SpatialStack-4B/5B** | Dual encoder; requires geometry encoder weights | Follow [SpatialStack](https://spatial-stack.github.io/) setup |

### APC full setup (for future work)

The APC-inspired strategy in this notebook uses a simplified two-pass approach.
The full [APC framework](https://github.com/KAIST-Visual-AI-Group/APC-VLM)
(ICCV 2025) requires:
- `grounding-dino` for object detection
- `segment-anything` for segmentation
- Depth estimation model (e.g., Depth Anything v2)
- Scene abstraction → perspective transformation → re-prompting

For mental rotation, a full APC pipeline would:
1. Detect the shape and its asymmetric features in 3D
2. Normalize both options to a canonical orientation
3. Compare in the normalized space

### References

- SpinBench (ICLR 2026): Zhang et al., [arXiv:2509.25390](https://arxiv.org/abs/2509.25390)
- TransformEval (under review): "Do VLMs Rotate in Mind?" [OpenReview](https://openreview.net/pdf?id=up2LD7vVdW)
- SpatialThinker (NeurIPS 2025 WS): Batra et al., [arXiv:2511.07403](https://arxiv.org/abs/2511.07403) / [project](https://hunarbatra.com/SpatialThinker/)
- Spatial-SSRL (CVPR 2026): Liu et al., [arXiv:2510.27606](https://arxiv.org/abs/2510.27606) / [GitHub](https://github.com/InternLM/Spatial-SSRL)
- SpatialDreamer: Cao et al., [arXiv:2512.07733](https://arxiv.org/abs/2512.07733) / [GitHub](https://github.com/mengcaopku/SpatialDreamer)
- APC (ICCV 2025): Lee et al., [project](https://apc-vlm.github.io/) / [GitHub](https://github.com/KAIST-Visual-AI-Group/APC-VLM)
- STAR-R1: Li et al., [arXiv:2505.15804](https://arxiv.org/abs/2505.15804) / [GitHub](https://github.com/zongzhao23/STAR-R1)
- ViLaSR (NeurIPS 2025): Wu et al., [arXiv:2506.09965](https://arxiv.org/abs/2506.09965) / [GitHub](https://github.com/AntResearchNLP/ViLaSR)
- SpatialLadder (ICLR 2026): Li et al., [arXiv:2510.08531](https://arxiv.org/abs/2510.08531)
- SpatialReasoner-R1 (NeurIPS 2025): Shen et al., [arXiv:2506.21656](https://arxiv.org/abs/2506.21656)
- SpaceThinker: [remyxai/SpaceThinker-Qwen2.5VL-3B](https://huggingface.co/remyxai/SpaceThinker-Qwen2.5VL-3B)
- SpaceOm: [remyxai/SpaceOm](https://huggingface.co/remyxai/SpaceOm)
- SpatialVLM (CVPR 2024): Chen et al., [arXiv:2401.12168](https://arxiv.org/abs/2401.12168)
- VLM² (2025): Liu et al., [arXiv:2511.20644](https://arxiv.org/abs/2511.20644) / [project](https://sairlab.org/vlm2/)
- SpatialStack (CVPR 2026): Zhang et al., [project](https://spatial-stack.github.io/)
- Awesome Spatial Intelligence in VLMs: [GitHub](https://github.com/mll-lab-nu/Awesome-Spatial-Intelligence-in-VLM)